In [4]:
import cv2
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tqdm import tqdm

In [5]:
BASE_DIR = "data"
SUB_DIRS = ["Train", "Validation", "Test"] 
CLASSES = ["Mask", "NonMask"]

# Target image size for model input (MobileNetV2 standard)
TARGET_SIZE = (224, 224)

# Augmentation settings - CONSERVATIVE for faces
TARGET_COUNT_TRAIN = 1500  # Only augment training data
AUGMENT_VALIDATION = False  # Never augment validation
AUGMENT_TEST = False  # Never augment test

In [6]:
# Face-specific augmentation (more conservative than general images)
train_datagen = ImageDataGenerator(
    rotation_range=15,           # Reduced: faces are usually upright
    width_shift_range=0.1,       # Slight horizontal shift
    height_shift_range=0.05,     # Minimal vertical shift
    shear_range=0.0,             # No shear - distorts facial features
    zoom_range=0.1,              # Slight zoom
    horizontal_flip=True,        # Yes - faces can be mirrored
    fill_mode='nearest',
    brightness_range=[0.8, 1.2], # Lighting variations
    # No vertical flip - faces don't appear upside down
)

In [7]:
def count_images_in_folder(folder_path):
    if not os.path.exists(folder_path):
        return 0
    return len([f for f in os.listdir(folder_path) 
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

def resize_images(folder_path, target_size=TARGET_SIZE):
    """
    Resize all images in a folder to target size
    Creates a backup of originals in 'original_backup' subfolder
    """
    if not os.path.exists(folder_path):
        print(f"Skipping {folder_path} (Folder not found)")
        return
    
    # Create backup folder
    backup_folder = os.path.join(folder_path, "original_backup")
    os.makedirs(backup_folder, exist_ok=True)
    
    image_files = [f for f in os.listdir(folder_path) 
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    print(f"\nResizing images in: {folder_path}")
    resized_count = 0
    
    for img_name in tqdm(image_files, desc="Resizing"):
        img_path = os.path.join(folder_path, img_name)
        
        # Skip if already in backup folder
        if 'original_backup' in img_path:
            continue
            
        try:
            # Read image
            img = cv2.imread(img_path)
            if img is None:
                print(f"  Warning: Could not read {img_name}")
                continue
            
            # Check if already correct size
            if img.shape[:2] == target_size:
                continue
            
            # Backup original (only if not already backed up)
            backup_path = os.path.join(backup_folder, img_name)
            if not os.path.exists(backup_path):
                cv2.imwrite(backup_path, img)
            
            # Resize
            img_resized = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
            
            # Save resized version
            cv2.imwrite(img_path, img_resized)
            resized_count += 1
            
        except Exception as e:
            print(f"  Error processing {img_name}: {e}")
    
    print(f"  Resized {resized_count} images to {target_size}")

def augment_folder(folder_path, target_count, datagen):
    """
    Generate augmented images until target count is reached
    Only creates new files, doesn't modify existing ones
    """
    if not os.path.exists(folder_path):
        print(f"Skipping {folder_path} (Folder not found)")
        return

    # Get existing images (exclude backup folder)
    existing_files = [f for f in os.listdir(folder_path) 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg'))
                      and not f.startswith('aug_')]  # Only count original images
    
    current_count = len(existing_files)
    print(f"\nProcessing: {folder_path}")
    print(f"  - Original images: {current_count}")
    
    if current_count >= target_count:
        print("  - Target count reached. Skipping augmentation.")
        return

    needed = target_count - current_count
    print(f"  - Generating {needed} augmented images...")

    generated_count = 0
    cycle = 0
    
    # Keep cycling through original images until we have enough
    while generated_count < needed:
        for file_name in existing_files:
            if generated_count >= needed:
                break
                
            img_path = os.path.join(folder_path, file_name)
            
            try:
                # Load image
                img = load_img(img_path, target_size=TARGET_SIZE)
                x = img_to_array(img)
                x = x.reshape((1,) + x.shape)
                
                # Generate one augmented version
                aug_filename = f"aug_{cycle}_{generated_count}_{file_name}"
                
                for batch in datagen.flow(x, batch_size=1,
                                         save_to_dir=folder_path,
                                         save_prefix=f'aug_{cycle}_{generated_count}',
                                         save_format='jpg'):
                    generated_count += 1
                    break  # Only generate one per loop
                    
            except Exception as e:
                print(f"    Error augmenting {file_name}: {e}")
                continue
        
        cycle += 1
        
        # Safety break to avoid infinite loop
        if cycle > 100:
            print(f"    Warning: Stopped after 100 cycles. Generated {generated_count}/{needed}")
            break

    # Final count
    total_images = len([f for f in os.listdir(folder_path) 
                       if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print(f"  - Done! Total images now: {total_images}")

In [8]:
def preprocess_dataset():
    
    #Resize all images   
    for sub_dir in SUB_DIRS:
        for category in CLASSES:
            folder_path = os.path.join(BASE_DIR, sub_dir, category)
            resize_images(folder_path, TARGET_SIZE)
    
    #Augment training data only
    for sub_dir in SUB_DIRS:
        # Skip validation and test
        if sub_dir == "Validation" and not AUGMENT_VALIDATION:
            print(f"\nSkipping {sub_dir} (Validation should not be augmented)")
            continue
        if sub_dir == "Test" and not AUGMENT_TEST:
            print(f"\nSkipping {sub_dir} (Test should not be augmented)")
            continue
            
        for category in CLASSES:
            folder_path = os.path.join(BASE_DIR, sub_dir, category)
            
            if sub_dir == "Train":
                augment_folder(folder_path, TARGET_COUNT_TRAIN, train_datagen)

    summary_data = []
    for sub_dir in SUB_DIRS:
        for category in CLASSES:
            folder_path = os.path.join(BASE_DIR, sub_dir, category)
            count = count_images_in_folder(folder_path)
            summary_data.append({
                'Split': sub_dir,
                'Class': category,
                'Count': count
            })
    
    df_summary = pd.DataFrame(summary_data)
    print("\n" + df_summary.to_string(index=False))
    print("PREPROCESSING COMPLETE!")


In [9]:
preprocess_dataset()


Resizing images in: data\Train\Mask


Resizing: 100%|██████████| 300/300 [00:07<00:00, 38.17it/s]


  Resized 300 images to (224, 224)

Resizing images in: data\Train\NonMask


Resizing: 100%|██████████| 300/300 [00:04<00:00, 74.09it/s]


  Resized 300 images to (224, 224)

Resizing images in: data\Validation\Mask


...

  Resized 153 images to (224, 224)

Resizing images in: data\Test\Mask


Resizing: 100%|██████████| 50/50 [00:02<00:00, 21.99it/s]


  Resized 50 images to (224, 224)

Resizing images in: data\Test\NonMask


Resizing: 100%|██████████| 50/50 [00:01<00:00, 44.53it/s]


  Resized 50 images to (224, 224)

Processing: data\Train\Mask
  - Original images: 300
  - Generating 1200 augmented images...
  - Done! Total images now: 1500

Processing: data\Train\NonMask
  - Original images: 300
  - Generating 1200 augmented images...
  - Done! Total images now: 1500

Skipping Validation (Validation should not be augmented)

Skipping Test (Test should not be augmented)

     Split   Class  Count
     Train    Mask   1500
     Train NonMask   1500
Validation    Mask    153
Validation NonMask    153
      Test    Mask     50
      Test NonMask     50
PREPROCESSING COMPLETE!


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# DATASET: folder structure
# data/Train/
#    Mask/
#    NonMask/

IMG_SIZE = 224
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    "data/Train/",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_data = train_datagen.flow_from_directory(
    "data/Validation/",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

# Load MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  

# Phase 1: feature extraction
# Add classifier head

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.4)(x)
output = Dense(2, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(
    optimizer=Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train Phase 1 (freeze)
model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# Phase 2: Unfreeze top layers (fine-tuning)
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False  # only unfreeze last 40 layers

model.compile(
    optimizer=Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

# Save model
model.save("mask_detectorV2.h5")
print("Model saved as mask_detectorV2.h5")


# END OF FACE MASK DETECTION